In [1]:
import polars as pl 
import geopandas as gpd
import polars as pl 
import duckdb
import os 
import polars as pl 
from cpom.masks.masks import Mask

os.environ['CPOM_SOFTWARE_DIR'] = "/home/willisc3/cpom_software2"
# from cpom.areas.area_plot import Polarplot
from cpom.altimetry.datasets.parquet_tools.spatio_temporal import (ParquetFilter,
                                                    #polygon_from_mask, 
                                                    get_db_connection, 
                                                    reproject, 
                                                    to_geometry
                                                    )

In [2]:
old_grid = pl.read_parquet("/home/willisc3/luna/CPOM/willisc3/SEC/greenland_cs2_5km/gridded_altimetry/**/*.parquet")

In [3]:
old_grid

latitude,longitude,beams,x,y,elevation,time,power,ascending,mode,x_bin,y_bin,x_cell_offset,y_cell_offset,year,month,x_part,y_part
f64,f64,null,f32,f32,f32,f32,f32,bool,f64,i64,i64,f32,f32,i32,i32,i64,i64
67.19716,313.033326,null,-85863.515625,-2.5005e6,1893.466187,6.18161984e8,3.935715,false,1.0,182,199,1636.487061,1991.380371,2010,8,0,0
67.19577,313.035741,null,-85763.507812,-2.5007e6,1894.20166,6.18161984e8,3.766089,false,1.0,182,199,1736.493164,1831.376953,2010,8,0,0
67.193994,313.035898,null,-85763.507812,-2.5009e6,1894.581787,6.18161984e8,3.697589,false,1.0,182,199,1736.493164,1631.369995,2010,8,0,0
67.185115,313.036682,null,-85763.5,-2.5019e6,1896.496216,6.18161984e8,3.358551,false,1.0,182,199,1736.499146,631.325195,2010,8,0,0
67.178961,313.041805,null,-85563.492188,-2.5026e6,1899.509644,6.18161984e8,3.3611,false,1.0,182,199,1936.505249,-68.692024,2010,8,0,0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
79.973451,335.597845,null,383053.0,-1.0192e6,549.439087,1.0748e9,-10.195558,true,3.0,276,496,553.002808,-1712.376587,2025,1,1,2
79.97634,335.599895,null,382978.53125,-1.0189e6,547.602112,1.0748e9,-5.61592,true,3.0,276,496,478.538574,-1403.567261,2025,1,1,2
79.980634,335.637189,null,383476.53125,-1.0182e6,531.005615,1.0748e9,-6.213318,true,3.0,276,496,976.528015,-715.57074,2025,1,1,2


In [5]:
new_grid = pl.read_parquet("/home/willisc3/luna/CPOM/willisc3/SEC1/greenland_cs2_5km/gridded_altimetry/**/*.parquet")
new_grid

latitude,longitude,elevation,elevation_err,power,mode,x,y,source,idx,ascending,datetime,month,day,h3_cell_id,x_bin,y_bin,x_cell_offset,y_cell_offset,year,x_part,y_part
f32,f32,f32,f32,f32,i8,f64,f64,str,i32,bool,datetime[μs],i8,i8,u64,i64,i64,f64,f64,i32,i32,i32
76.403015,293.535309,613.370117,0.527164,-2.130173,3,-541425.347319,-1.3770e6,"""CS_OFFL_SIR_TDP_LI_GREENL_2010…",2549,false,2010-12-20 10:18:34.474174,12,20,608028124210266111,91,424,1074.652681,523.085893,2010,0,2
76.278221,294.721588,330.064301,0.483357,-9.928377,3,-517551.79175,-1.4007e6,"""CS_OFFL_SIR_TDP_LI_GREENL_2010…",233,true,2010-11-28 22:12:39.787446,11,28,608028123673395199,96,419,-51.79175,1753.347827,2010,0,2
76.173737,294.531525,373.896088,1.935789,-5.361217,3,-526208.914313,-1.4098e6,"""CS_OFFL_SIR_TDP_LI_GREENL_2010…",2709,false,2010-09-24 14:33:51.245269,9,24,608028123270742015,94,418,1291.085687,-2274.490412,2010,0,2
76.173874,294.530792,373.686859,1.935789,-9.745895,3,-526221.525155,-1.4098e6,"""CS_OFFL_SIR_TDP_LI_GREENL_2010…",2638,false,2010-12-18 10:21:04.911467,12,18,608028123270742015,94,418,1278.474845,-2253.523271,2010,0,2
76.400284,295.43692,461.70636,0.958695,-4.041195,3,-495535.53715,-1.3945e6,"""CS_OFFL_SIR_TDP_LI_GREENL_2010…",262,true,2010-10-28 23:40:11.154224,10,28,608101828147544063,100,421,1964.46285,-1967.338096,2010,0,2
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
75.092117,337.54834,1523.778809,1.290844,-15.583529,3,622642.333385,-1.4996e6,"""CS_OFFL_SIR_TDP_LI_GREENL_2025…",1858,true,2025-01-28 02:51:00.116835,1,28,608120663911170047,324,400,142.333385,-2111.37954,2025,2,2
75.093384,337.536438,1519.544678,1.622565,-10.595819,3,622276.959129,-1.4996e6,"""CS_OFFL_SIR_TDP_LI_GREENL_2025…",1884,true,2025-01-03 04:04:29.765408,1,3,608120663911170047,324,400,-223.040871,-2111.646985,2025,2,2
75.095444,337.519012,1502.846191,2.003153,-6.621678,3,621734.244746,-1.4996e6,"""CS_OFFL_SIR_TDP_LI_GREENL_2025…",1885,true,2025-01-03 04:04:29.810530,1,3,608120663911170047,324,400,-765.755254,-2091.824743,2025,2,2


In [2]:
from cpom.gridding.gridareas import GridArea  # local import: optional dependency   
this_grid_area = GridArea("greenland", 5000)

In [14]:
new_ds_path = "/home/willisc3/luna/CPOM/archive/SATS/_PARQUET_STORE/north/cs2/CryoTEMPO_baseline_D/**/*.parquet"

new_ds = (
    ParquetFilter(new_ds_path, data_crs=3413, engine = "duckdb")
    .get_cpom_grid_mask(mask=Mask("greenland_icesheet_2km_grid_mask"))
    .get_grid_cells_from_grid_area(GridArea("greenland", 5000))
    .custom_clause(
        "date_diff('second', TIMESTAMP '1991-01-01 00:00:00', datetime) / (365.25 * 86400) AS time"
    )
    .run(output_path="tmp.parquet")
)

# pf2 = (
#     ParquetFilter(new_ds, data_crs=3413, engine="duckdb")
#     .select("MIN(time)", "MAX(time)"])
#     .run()
# )


In [15]:
new_ds

In [6]:
# new_ds is a TableHandle(con, table_name="tmp")

# arbitrary SQL against the table -> DuckDB relation
result = new_ds.sql("SELECT * FROM tmp LIMIT 10")
result.show()

# full table as a DataFrame
df = new_ds.pl()   # polars
df = new_ds.df()   # pandas

# or write your own query via .sql(), lazily evaluated until you call
# .pl()/.df()/.show() on the returned relation
summary = new_ds.sql("""
    SELECT x_bin, y_bin, count(*) AS n
    FROM tmp
    GROUP BY x_bin, y_bin
""").pl()


┌───────────┬───────────┬────────────┬───────────────┬─────────────┬──────┬────────────────────┬─────────────────────┬─────────────────────────────────────────────────────────────────────────┬───────┬───────────┬────────────────────────────┬───────┬───────┬──────┬────────────────────┬───────┬───────┬─────────────────────┬─────────────────────┐
│ latitude  │ longitude │ elevation  │ elevation_err │    power    │ mode │         x          │          y          │                                 source                                  │  idx  │ ascending │          datetime          │ year  │ month │ day  │     h3_cell_id     │ x_bin │ y_bin │    x_cell_offset    │    y_cell_offset    │
│   float   │   float   │   float    │     float     │    float    │ int8 │       double       │       double        │                                 varchar                                 │ int32 │  boolean  │         timestamp          │ int64 │ int8  │ int8 │       uint64       │ int64 │ int64 │       

In [ ]:
# .run(output_table="tmp") now returns a TableHandle bound to its own
# DuckDB connection, so `new_ds` is the handle, not the filter instance.

# query it as a DuckDB relation
result = new_ds.sql("SELECT * FROM tmp LIMIT 10")
result.show()

# or pull it into polars/pandas
df = new_ds.pl()   # polars
df = new_ds.df()   # pandas

# aggregate example
summary = new_ds.sql("""
    SELECT x_bin, y_bin, count(*) AS n
    FROM tmp
    GROUP BY x_bin, y_bin
""").pl()
